# 🔬 HAM10000 Production Model Training: EfficientNet-B4 + Zero-Leakage + Calibration + Dual-Output ONNX
### DermAssist AI Medical Core Training Engine
---
**Platform**: Kaggle GPU (NVIDIA Tesla T4 x2 or P100)
**Dataset**: `kmader/skin-cancer-mnist-ham10000`



### 🎯 Production Invariants Guaranteed:
1. **Zero Patient-Level Data Leakage**: Lesions grouped strictly by `lesion_id` using `StratifiedGroupKFold`.
2. **Class Imbalance Mitigation**: Class-weighted Cross-Entropy addressing 67% Melanocytic Nevus (`nv`) prevalence.
3. **Compound Scaling**: EfficientNet-B4 operating at 380x380 resolution.
4. **Post-Hoc Temperature Calibration**: Optimization of $T^*$ via validation NLL to minimize Expected Calibration Error (ECE).
5. **Dual-Output ONNX Export**: Emits both classification `logits` (7,) and convolutional `features` (1792, H, W) for live Grad-CAM explainability.


In [ ]:
# 0. Production-safe retraining guard
# This cell runs before the original notebook cells and prevents CPU quota waste.
import os, json, base64, zipfile
import torch
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix

assert torch.cuda.is_available(), 'GPU is not enabled; enable a T4/P100 accelerator before running.'
print(f'Preflight GPU: {torch.cuda.get_device_name(0)} | CUDA {torch.version.cuda}')

def _production_guard(_result):
    g = globals()
    # The original val_df is a lesion-grouped holdout. Split it into model-selection,
    # calibration, and untouched final-test partitions.
    if 'train_df' in g and 'val_df' in g and not g.get('_safe_split_done'):
        hold = g['val_df'].copy()
        outer = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=123)
        sel_i, rem_i = next(outer.split(hold, hold['label'], groups=hold['lesion_id']))
        selection = hold.iloc[sel_i].reset_index(drop=True)
        remaining = hold.iloc[rem_i].reset_index(drop=True)
        inner = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=456)
        cal_i, test_i = next(inner.split(remaining, remaining['label'], groups=remaining['lesion_id']))
        g['val_df'] = selection
        g['calibration_df'] = remaining.iloc[cal_i].reset_index(drop=True)
        g['test_df'] = remaining.iloc[test_i].reset_index(drop=True)
        parts = [g['train_df'], g['val_df'], g['calibration_df'], g['test_df']]
        groups = [set(p['lesion_id']) for p in parts]
        assert all(not (groups[i] & groups[j]) for i in range(4) for j in range(i + 1, 4)), 'CRITICAL: lesion overlap'
        g['_safe_split_done'] = True
        print(f"Lesion-safe split: train={len(parts[0])}, selection={len(parts[1])}, calibration={len(parts[2])}, test={len(parts[3])}")

    # Create independent loaders, but leave the original selection loader active.
    if g.get('_safe_split_done') and 'HAM10000Dataset' in g and 'val_transform' in g and not g.get('_safe_loaders_done'):
        D, C = g['DataLoader'], g['HAM10000Dataset']
        args = dict(batch_size=g['BATCH_SIZE'], shuffle=False, num_workers=4, pin_memory=True)
        g['calibration_loader'] = D(C(g['calibration_df'], g['val_transform']), **args)
        g['test_loader'] = D(C(g['test_df'], g['val_transform']), **args)
        g['_safe_loaders_done'] = True
        print('Independent calibration/test loaders created; selection loader remains active.')

    # Only after cell 8 has selected best_epoch may calibration replace val_loader.
    if g.get('_safe_loaders_done') and g.get('best_epoch', 0) > 0 and not g.get('_calibration_armed'):
        g['val_loader'] = g['calibration_loader']
        g['_calibration_armed'] = True
        print('Calibration armed after selection-split model selection.')

    # After cell 9, evaluate the untouched test split with the calibrated model.
    if g.get('_calibration_armed') and 'optimal_t' in g and not g.get('_test_done'):
        model, device = g['model'], g['device']
        model.eval(); logits, labels = [], []
        with torch.no_grad():
            for images, y in g['test_loader']:
                z, _ = model(images.to(device, non_blocking=True))
                logits.append(z.float().cpu().numpy()); labels.extend(y.numpy())
        z, y = np.concatenate(logits), np.asarray(labels)
        temperature = max(0.05, float(g['optimal_t']))
        scaled = z / temperature
        probs = np.exp(scaled - scaled.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
        pred = probs.argmax(1)
        report = classification_report(y, pred, target_names=g['CLASS_NAMES'], output_dict=True, zero_division=0)
        metrics = {
            'split': 'lesion-disjoint-final-test',
            'temperature': temperature,
            'balanced_accuracy': float(balanced_accuracy_score(y, pred)),
            'macro_f1': float(f1_score(y, pred, average='macro', zero_division=0)),
            'weighted_f1': float(f1_score(y, pred, average='weighted', zero_division=0)),
            'classification_report': report,
            'confusion_matrix': confusion_matrix(y, pred).tolist(),
            'malignant_recall': {c: report.get(c, {}).get('recall') for c in ('bcc', 'mel')},
        }
        with open('/kaggle/working/final_test_metrics.json', 'w') as f: json.dump(metrics, f, indent=2)
        print(f"FINAL TEST | BalAcc={metrics['balanced_accuracy']:.4f} Macro-F1={metrics['macro_f1']:.4f} BCC recall={metrics['malignant_recall']['bcc']:.4f} MEL recall={metrics['malignant_recall']['mel']:.4f}")
        g['_test_done'] = True

    # Export the exact head from the same model as ONNX, with runtime-compatible temperature.
    if 'model' in g and 'optimal_t' in g:
        head = g['model'].backbone.classifier[1]
        weights = head.weight.detach().cpu().numpy().astype('<f4')
        payload = {
            'classes': g['CLASS_NAMES'], 'shape': list(weights.shape),
            'weight_f32_b64': base64.b64encode(weights.tobytes()).decode('ascii'),
            'bias': head.bias.detach().cpu().numpy().astype('float32').tolist(),
            'temperature': round(float(g['optimal_t']), 6),
            'note': 'Matching trained head; temperature fit on independent calibration split.'
        }
        with open('/kaggle/working/head_v2.json', 'w') as f: json.dump(payload, f, indent=2)

    bundle = '/kaggle/working/model_files.zip'
    if os.path.exists(bundle) and os.path.exists('/kaggle/working/head_v2.json'):
        with zipfile.ZipFile(bundle, 'a', compression=zipfile.ZIP_DEFLATED) as z:
            if 'head_v2.json' not in z.namelist(): z.write('/kaggle/working/head_v2.json', 'head_v2.json')

get_ipython().events.register('post_run_cell', _production_guard)


In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
import glob
import json
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, f1_score, balanced_accuracy_score, confusion_matrix
from scipy.optimize import minimize_scalar

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

# Verify GPU hardware
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using compute device: {device}')
if torch.cuda.is_available():
    print(f'GPU Model: {torch.cuda.get_device_name(0)}')
    print(f'Device Count: {torch.cuda.device_count()}')


In [ ]:
# 2. Configuration & Class Taxonomy
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
CLASS_TO_IDX = {name: idx for idx, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {idx: name for idx, name in enumerate(CLASS_NAMES)}

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_SIZE = (380, 380) # EfficientNet-B4 native resolution
BATCH_SIZE = 32
EPOCHS = 10
LR = 3e-4


In [ ]:
# 3. Dataset Discovery & Indexing (Robust Auto-Detection + kagglehub Fallback)
import glob
import os
import pandas as pd

# 1. Attempt primary discovery under /kaggle/input
csv_matches = glob.glob("/kaggle/input/**/HAM10000_metadata.csv", recursive=True)

# 2. Fallback to kagglehub if not found under /kaggle/input
if not csv_matches:
    print("HAM10000 metadata CSV not found in /kaggle/input. Initiating kagglehub download...")
    try:
        import kagglehub
        downloaded_dir = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
        print(f"kagglehub successfully downloaded dataset to: {downloaded_dir}")
    except Exception as e:
        raise RuntimeError(f"kagglehub failed to download kmader/skin-cancer-mnist-ham10000: {e}") from e

    if not downloaded_dir or not os.path.exists(downloaded_dir):
        raise FileNotFoundError(f"kagglehub target directory does not exist: {downloaded_dir}")

    csv_matches = glob.glob(os.path.join(downloaded_dir, "**/HAM10000_metadata.csv"), recursive=True)
    if not csv_matches:
        # Also check root of downloaded_dir
        direct_csv = os.path.join(downloaded_dir, "HAM10000_metadata.csv")
        if os.path.exists(direct_csv):
            csv_matches = [direct_csv]
        else:
            raise FileNotFoundError(f"HAM10000_metadata.csv not found in kagglehub directory: {downloaded_dir}")

csv_path = csv_matches[0]
base_dir = os.path.dirname(csv_path)
print(f"Verified metadata path: {csv_path}")
print(f"Verified dataset base directory: {base_dir}")

# 3. Auto-detect image directories (part 1, part 2, or images)
image_dirs = [
    d for d in glob.glob(os.path.join(base_dir, "**"), recursive=True)
    if os.path.isdir(d) and ("part_1" in d.lower() or "part_2" in d.lower() or "images" in d.lower())
]
if not image_dirs:
    image_dirs = [base_dir]
print(f"Discovered candidate image directories: {image_dirs}")

# 4. Load metadata and map image paths
df = pd.read_csv(csv_path)
print(f"Total raw metadata records: {len(df)}")
if len(df) == 0:
    raise ValueError(f"HAM10000_metadata.csv at {csv_path} contains 0 records!")

image_path_map = {}
for img_dir in image_dirs:
    for ext in ("*.jpg", "*.jpeg", "*.png"):
        for file_path in glob.glob(os.path.join(img_dir, ext)):
            image_id = os.path.splitext(os.path.basename(file_path))[0]
            image_path_map[image_id] = file_path

print(f"Found {len(image_path_map)} image files across candidate directories.")
df["file_path"] = df["image_id"].map(image_path_map)
df = df.dropna(subset=["file_path"]).copy()

if len(df) == 0:
    raise FileNotFoundError(f"Zero matching image files found for metadata IDs in {image_dirs}")

df["label"] = df["dx"].map(CLASS_TO_IDX)
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

print(f"Successfully indexed {len(df)} images across {df['lesion_id'].nunique()} unique lesions.")
assert len(df) >= 5000, f"Expected at least 5000 images in HAM10000 dataset, found only {len(df)}"
df.head()


In [ ]:
# 4. Zero Patient Leakage Splitting via StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in sgkf.split(df, df['label'], groups=df['lesion_id']):
    train_df = df.iloc[train_idx].copy().reset_index(drop=True)
    val_df = df.iloc[val_idx].copy().reset_index(drop=True)
    break

# Mathematically assert zero lesion overlap
train_lesions = set(train_df['lesion_id'].unique())
val_lesions = set(val_df['lesion_id'].unique())
overlap = train_lesions.intersection(val_lesions)
assert len(overlap) == 0, f'CRITICAL LEAKAGE DETECTED: {len(overlap)} lesions overlap!'
print('✅ Zero Leakage Confirmed!')
print(f'Train Set: {len(train_df)} images ({len(train_lesions)} lesions)')
print(f'Val Set:   {len(val_df)} images ({len(val_lesions)} lesions)')


In [ ]:
# 5. Class Distribution & Balanced Loss Weight Computation
class_counts = train_df['label'].value_counts().sort_index().values
total_samples = len(train_df)
raw_weights = total_samples / (len(CLASS_NAMES) * class_counts.astype(np.float32))
# Square-root damping to prevent gradient explosion on rare classes
damped_weights = np.sqrt(raw_weights)
damped_weights = damped_weights / damped_weights.sum() * len(CLASS_NAMES)
weight_tensor = torch.tensor(damped_weights, dtype=torch.float32).to(device)

for name, count, w in zip(CLASS_NAMES, class_counts, damped_weights):
    print(f'Class {name:6s} | Count: {count:5d} ({count/total_samples*100:5.2f}%) | Weight: {w:.3f}')


In [ ]:
# 6. PyTorch Dataset & DataLoaders
class HAM10000Dataset(Dataset):
    def __init__(self, df: pd.DataFrame, transform=None):
        self.df = df
        self.transform = transform
        self.file_paths = df['file_path'].values
        self.labels = df['label'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.file_paths[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])

train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset = HAM10000Dataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
# 7. Dual-Output EfficientNet-B4 Model Architecture
class DualOutputEfficientNetB4(nn.Module):
    def __init__(self, num_classes=7, pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B4_Weights.DEFAULT if pretrained else None
        self.backbone = models.efficientnet_b4(weights=weights)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(
            nn.Dropout(p=0.4, inplace=True),
            nn.Linear(in_features=in_features, out_features=num_classes)
        )

    def forward(self, x):
        features = self.backbone.features(x)
        pooled = self.backbone.avgpool(features)
        flattened = torch.flatten(pooled, 1)
        logits = self.backbone.classifier(flattened)
        return logits, features

model = DualOutputEfficientNetB4(num_classes=7, pretrained=True).to(device)
criterion = nn.CrossEntropyLoss(weight=weight_tensor)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())


In [ ]:
# 8. Training & Validation Execution Loop
best_macro_f1 = 0.0
best_epoch = 0
best_weights_path = '/kaggle/working/best_model.pt'

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    running_loss, train_correct, total_train = 0.0, 0, 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            logits, _ = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        train_correct += (torch.argmax(logits, dim=1) == labels).sum().item()
        total_train += labels.size(0)

    scheduler.step()
    train_loss = running_loss / total_train
    train_acc = train_correct / total_train

    # Validation
    model.eval()
    val_loss = 0.0
    all_val_logits, all_val_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                logits, _ = model(images)
                loss = criterion(logits, labels)
            val_loss += loss.item() * images.size(0)
            all_val_logits.append(logits.cpu().numpy())
            all_val_labels.extend(labels.cpu().numpy())

    val_loss = val_loss / len(val_dataset)
    val_logits_np = np.concatenate(all_val_logits, axis=0)
    val_labels_np = np.array(all_val_labels)
    val_preds_np = np.argmax(val_logits_np, axis=1)
    val_acc = np.mean(val_preds_np == val_labels_np)
    val_balanced_acc = balanced_accuracy_score(val_labels_np, val_preds_np)
    val_macro_f1 = f1_score(val_labels_np, val_preds_np, average='macro', zero_division=0)
    elapsed = time.time() - start_time

    print(f'Epoch [{epoch:02d}/{EPOCHS:02d}] ({elapsed:.1f}s) | Train Loss: {train_loss:.4f} Acc: {train_acc*100:.1f}% | ' 
          f'Val Loss: {val_loss:.4f} Acc: {val_acc*100:.1f}% BalAcc: {val_balanced_acc*100:.1f}% Macro-F1: {val_macro_f1:.4f}')

    if val_macro_f1 > best_macro_f1:
        best_macro_f1 = val_macro_f1
        best_epoch = epoch
        torch.save(model.state_dict(), best_weights_path)
        print(f"Saved best checkpoint: {best_weights_path} at epoch {epoch}")


In [ ]:
# 9. Temperature Calibration Optimization (ECE Minimization)
model.load_state_dict(torch.load(best_weights_path, map_location=device))
model.eval()

final_val_logits, final_val_labels = [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        logits, _ = model(images)
        final_val_logits.append(logits.cpu().numpy())
        final_val_labels.extend(labels.numpy())

val_logits_np = np.concatenate(final_val_logits, axis=0)
val_labels_np = np.array(final_val_labels)

def softmax_np(x):
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp_vals = np.exp(shifted)
    return exp_vals / np.sum(exp_vals, axis=-1, keepdims=True)

def compute_ece(probs, labels, n_bins=10):
    confs = np.max(probs, axis=1)
    preds = np.argmax(probs, axis=1)
    accs = (preds == labels)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (confs > bins[i]) & (confs <= bins[i+1])
        if np.sum(mask) > 0:
            ece += (np.sum(mask) / len(labels)) * np.abs(np.mean(accs[mask]) - np.mean(confs[mask]))
    return float(ece)

def nll_func(T):
    scaled = val_logits_np / max(0.05, float(T))
    probs = np.clip(softmax_np(scaled), 1e-12, 1.0 - 1e-12)
    return float(-np.sum(np.log(probs[np.arange(len(probs)), val_labels_np])) / len(probs))

raw_nll = nll_func(1.0)
raw_ece = compute_ece(softmax_np(val_logits_np), val_labels_np)
res = minimize_scalar(nll_func, bounds=(0.1, 10.0), method='bounded')
optimal_t = float(res.x)
calibrated_nll = nll_func(optimal_t)
calibrated_ece = compute_ece(softmax_np(val_logits_np / optimal_t), val_labels_np)

print(f'Raw Val NLL:         {raw_nll:.4f}')
print(f'Calibrated Val NLL:  {calibrated_nll:.4f}')
print(f'Uncalibrated Raw ECE: {raw_ece*100:.2f}%')
print(f'Calibrated ECE:       {calibrated_ece*100:.2f}%')
print(f'Optimal Temperature:  T* = {optimal_t:.4f}')


In [ ]:
# 10. Clinical Classification Report
from sklearn.metrics import confusion_matrix

val_preds_np = np.argmax(val_logits_np, axis=1)
macro_f1 = f1_score(val_labels_np, val_preds_np, average='macro', zero_division=0)
weighted_f1 = f1_score(val_labels_np, val_preds_np, average='weighted', zero_division=0)
cm = confusion_matrix(val_labels_np, val_preds_np)

print('\n--- CLINICAL VALIDATION CLASSIFICATION REPORT ---')
print(f'Best Validation Epoch: {best_epoch} with Macro-F1: {best_macro_f1:.4f}')
print(f'Macro-F1:              {macro_f1:.4f}')
print(f'Weighted-F1:           {weighted_f1:.4f}')
print(f'Validation NLL (raw):  {raw_nll:.4f}')
print(f'Validation NLL (cal):  {calibrated_nll:.4f}')
print(f'Raw ECE:               {raw_ece*100:.2f}%')
print(f'Calibrated ECE:        {calibrated_ece*100:.2f}%')
print(f'Optimal Temperature:   T* = {optimal_t:.4f}')
print('\nPer-Class Detailed Report:')
print(classification_report(val_labels_np, val_preds_np, target_names=CLASS_NAMES, digits=4))
print('\nConfusion Matrix (Rows=True, Columns=Predicted):')
cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
print(cm_df)


In [ ]:
# 11. Dual-Output ONNX Export
!pip install onnx onnxscript
model.cpu()
model.eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE[0], IMAGE_SIZE[1], dtype=torch.float32)
onnx_path = '/kaggle/working/efficientnet_b4_ham10000.onnx'

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['logits', 'features'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'logits': {0: 'batch_size'},
        'features': {0: 'batch_size'}
    }
)
print(f'✅ Successfully exported {onnx_path} ({os.path.getsize(onnx_path)/(1024*1024):.2f} MB)')

meta = {
    'model_architecture': 'EfficientNet-B4',
    'optimal_temperature': round(optimal_t, 4),
    'raw_ece': round(raw_ece, 4),
    'calibrated_ece': round(calibrated_ece, 4),
    'raw_nll': round(raw_nll, 4),
    'calibrated_nll': round(calibrated_nll, 4),
    'val_macro_f1': round(macro_f1, 4),
    'val_weighted_f1': round(weighted_f1, 4),
    'best_epoch': int(best_epoch),
    'class_names': CLASS_NAMES
}
calib_path = '/kaggle/working/calibration_params.json'
with open(calib_path, 'w') as f:
    json.dump(meta, f, indent=2)
print(f'✅ Saved {calib_path}')


In [ ]:
# 12. Artifact Validation & Packaging
import os
import zipfile
import onnx

onnx_path = '/kaggle/working/efficientnet_b4_ham10000.onnx'
calib_path = '/kaggle/working/calibration_params.json'
weights_path = '/kaggle/working/best_model.pt'

assert os.path.exists(onnx_path), f'ONNX model missing at {onnx_path}'
assert os.path.exists(calib_path), f'Calibration metadata missing at {calib_path}'
assert os.path.exists(weights_path), f'Checkpoint weights missing at {weights_path}'
onnx_data_path = onnx_path + '.data'
head_path = '/kaggle/working/head_v2.json'
assert os.path.exists(onnx_data_path), f'ONNX external data missing at {onnx_data_path}'
assert os.path.exists(head_path), f'Runtime classifier head missing at {head_path}'

# Verify structural validity of ONNX model
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print('✅ ONNX model structurally validated.')
print('  Input tensors: ', [x.name for x in onnx_model.graph.input])
print('  Output tensors:', [x.name for x in onnx_model.graph.output])

# Package into zip bundle
zip_path = '/kaggle/working/model_files.zip'
artifacts = [weights_path, onnx_path, onnx_data_path, calib_path, head_path]

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in artifacts:
        z.write(path, arcname=os.path.basename(path))
        print(f'  Added {os.path.basename(path)} ({os.path.getsize(path)/1024**2:.2f} MB)')

with zipfile.ZipFile(zip_path) as z:
    required = {os.path.basename(p) for p in artifacts}
    assert required.issubset(set(z.namelist())), 'Bundle is missing a required artifact.'
print(f'✅ Successfully generated {zip_path} ({os.path.getsize(zip_path)/1024**2:.2f} MB)')
